# Analysing Team Performance - Batch Processing - Bronze Layer

To ensure data integrity and prevent issues downstream, we follow the Medallion architecture for data organization, transformation, and consumption.

The **Bronze Layer** is the first step in our batch processing workflow. It acts as the raw data repository where all incoming data is initially stored. The primary role of this layer is to ensure that data is captured accurately and remains available for further processing.

### Ingested Raw Data Sources
- **Teams**
- **Squads**
- **Goals**
- **Penalty Kicks**
- **Substitutions**
- **Bookings**


In [0]:
%pip uninstall -y databricks_helpers
%pip install git+https://github.com/data-derp/databricks_helpers#egg=databricks_helpers

In [0]:
exercise_name = "team_analysis_batch_processing/bronze"

In [0]:
from databricks_helpers.databricks_helpers import DataDerpDatabricksHelpers

helpers = DataDerpDatabricksHelpers(dbutils, exercise_name)

current_user = helpers.current_user()
working_directory = helpers.working_directory()

print(f"Your current working directory is: {working_directory}")

helpers.clean_working_directory()

### **Teams**

In [0]:
url = "https://raw.githubusercontent.com/jfjelstul/worldcup/refs/heads/master/data-csv/teams.csv"
display(working_directory)

helpers.download_to_local_dir(url)
filepath = f"{working_directory}/teams.csv" 
dbutils.fs.ls(f"{working_directory}")

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

def create_source(filepath: str) -> DataFrame:
    custom_schema = StructType([
        StructField("key_id", IntegerType(), True),
        StructField("team_id", StringType(), True),
        StructField("team_name", StringType(), True),
        StructField("team_code", StringType(), True),
        StructField("mens_team", IntegerType(), True),
        StructField("womens_team", IntegerType(), True),
        StructField("federation_name", StringType(), True),
        StructField("region_name", StringType(), True),
        StructField("confederation_id", StringType(), True),
        StructField("confederation_name", StringType(), True),
        StructField("confederation_code", StringType(), True),
        StructField("mens_team_wikipedia_link", StringType(), True),
        StructField("womens_team_wikipedia_link", StringType(), True),
        StructField("federation_wikipedia_link", StringType(), True),
    ])
    
    df = spark.read.format("csv") \
        .option("header", True) \
        .option("delimiter", ",") \
        .option("escape", "\\") \
        .schema(custom_schema) \
        .load(filepath)
    return df

teams_bronze_df = create_source(filepath)
teams_bronze_df.printSchema()
display(teams_bronze_df.count())
display(teams_bronze_df)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/teams"
    
### Put your code here.
    mode_name = "overwrite"
    
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)
    
    
write(teams_bronze_df)
dbutils.fs.ls(f"{working_directory}/output/teams")

### **Squads**

In [0]:
url = "https://raw.githubusercontent.com/jfjelstul/worldcup/refs/heads/master/data-csv/squads.csv"
helpers.download_to_local_dir(url)
filepath = f"{working_directory}/squads.csv" 
dbutils.fs.ls(f"{working_directory}")


In [0]:
def create_source(filepath: str) -> DataFrame:
    custom_schema = StructType([
        StructField("key_id", IntegerType(), True),
        StructField("tournament_id", StringType(), True),
        StructField("tournament_name", StringType(), True),
        StructField("team_id", StringType(), True),
        StructField("team_name", StringType(), True),
        StructField("team_code", StringType(), True),
        StructField("player_id", StringType(), True),
        StructField("family_name", StringType(), True),
        StructField("given_name", StringType(), True),
        StructField("shirt_number", IntegerType(), True),
        StructField("position_name", StringType(), True),
        StructField("position_code", StringType(), True)
    ])
    
    df = spark.read.format("csv") \
        .option("header", True) \
        .option("delimiter", ",") \
        .option("escape", "\\") \
        .schema(custom_schema) \
        .load(filepath)
    return df

squads_bronze_df = create_source(filepath)
squads_bronze_df.printSchema()
display(squads_bronze_df)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/squads"
    
### Put your code here.
    mode_name = "overwrite"
    
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)
    
    
write(teams_bronze_df)
dbutils.fs.ls(f"{working_directory}/output/squads")

### **Goals**

In [0]:
url = "https://raw.githubusercontent.com/jfjelstul/worldcup/refs/heads/master/data-csv/goals.csv"
helpers.download_to_local_dir(url)
filepath = f"{working_directory}/goals.csv" 
dbutils.fs.ls(f"{working_directory}")

In [0]:
from pyspark.sql import functions as F
def create_source(filepath: str) -> DataFrame:
    custom_schema = StructType([
        StructField("key_id", StringType(), True),
        StructField("goal_id", StringType(), True),
        StructField("tournament_id", StringType(), True),
        StructField("tournament_name", StringType(), True),
        StructField("match_id", StringType(), True),
        StructField("match_name", StringType(), True),
        StructField("match_date", DateType(), True),
        StructField("stage_name", StringType(), True),
        StructField("group_name", StringType(), True),
        StructField("team_id", StringType(), True),
        StructField("team_name", StringType(), True),
        StructField("team_code", StringType(), True),
        StructField("player_id", StringType(), True),
        StructField("family_name", StringType(), True),
        StructField("given_name", StringType(), True),
        StructField("shirt_number", IntegerType(), True),
        StructField("player_team_id", StringType(), True),
        StructField("player_team_name", StringType(), True),
        StructField("player_team_code", StringType(), True),
        StructField("minute_label", StringType(), True),
        StructField("minute_regulation", IntegerType(), True),
        StructField("minute_stoppage", IntegerType(), True),
        StructField("match_period", StringType(), True),
        StructField("own_goal", IntegerType(), True),
        StructField("penalty", IntegerType(), True),
    ])
    
    df = spark.read.format("csv") \
        .option("header", True) \
        .option("delimiter", ",") \
        .option("escape", "\\") \
        .schema(custom_schema) \
        .load(filepath)
    return df

goals_bronze_df = create_source(filepath)
goals_bronze_df.printSchema()
display(goals_bronze_df)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/goals"
    mode_name = "overwrite"
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)

write(goals_bronze_df)
dbutils.fs.ls(f"{working_directory}/output/goals")

### **Penalty **Kicks****

In [0]:
url = "https://raw.githubusercontent.com/jfjelstul/worldcup/refs/heads/master/data-csv/penalty_kicks.csv"
helpers.download_to_local_dir(url)
filepath = f"{working_directory}/penalty_kicks.csv"
dbutils.fs.ls(f"{working_directory}")

In [0]:
def create_source(filepath: str) -> DataFrame:
    custom_schema = StructType([
        StructField("key_id", IntegerType(), True),
        StructField("penalty_kick_id", StringType(), True),
        StructField("tournament_id", StringType(), True),
        StructField("tournament_name", StringType(), True),
        StructField("match_id", StringType(), True),
        StructField("match_name", StringType(), True),
        StructField("match_date", DateType(), True),
        StructField("stage_name", StringType(), True),
        StructField("group_name", StringType(), True),
        StructField("team_id", StringType(), True),
        StructField("team_name", StringType(), True),
        StructField("team_code", StringType(), True),
        StructField("home_team", IntegerType(), True),
        StructField("away_team", IntegerType(), True),
        StructField("player_id", StringType(), True),
        StructField("family_name", StringType(), True),
        StructField("given_name", StringType(), True),
        StructField("minute_regulation", IntegerType(), True),
        StructField("shirt_number", IntegerType(), True),
        StructField("converted", IntegerType(), True)
    ])
    
    df = spark.read.format("csv") \
        .option("header", True) \
        .option("delimiter", ",") \
        .option("escape", "\\") \
        .schema(custom_schema) \
        .load(filepath)
    return df

penalty_kicks_bronze_df = create_source(filepath)
penalty_kicks_bronze_df.printSchema()
display(penalty_kicks_bronze_df)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/penalty_kicks"
    mode_name = "overwrite"
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)

write(penalty_kicks_bronze_df)
dbutils.fs.ls(f"{working_directory}/output/penalty_kicks")

### **Substitutions**

In [0]:
url = "https://raw.githubusercontent.com/jfjelstul/worldcup/refs/heads/master/data-csv/substitutions.csv"
helpers.download_to_local_dir(url)
filepath = f"{working_directory}/substitutions.csv"
dbutils.fs.ls(f"{working_directory}")

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
def create_source(filepath: str) -> DataFrame:
    custom_schema = StructType([
        StructField("key_id", StringType(), True),
        StructField("substitution_id", StringType(), True),
        StructField("tournament_id", StringType(), True),
        StructField("tournament_name", StringType(), True),
        StructField("match_id", StringType(), True),
        StructField("match_name", StringType(), True),
        StructField("match_date", DateType(), True),
        StructField("stage_name", StringType(), True),
        StructField("group_name", StringType(), True),
        StructField("team_id", StringType(), True),
        StructField("team_name", StringType(), True),
        StructField("team_code", StringType(), True),
        StructField("player_id", StringType(), True),
        StructField("family_name", StringType(), True),
        StructField("given_name", StringType(), True),
        StructField("shirt_number", IntegerType(), True),
        StructField("player_team_id", StringType(), True),
        StructField("player_team_name", StringType(), True),
        StructField("player_team_code", StringType(), True),
        StructField("minute_label", StringType(), True),
        StructField("minute_regulation", IntegerType(), True),
        StructField("minute_stoppage", IntegerType(), True),
        StructField("match_period", StringType(), True),
        StructField("substitution_type", StringType(), True),
        StructField("going_off", IntegerType(), True),
        StructField("coming_on", IntegerType(), True),
    ])
    
    df = spark.read.format("csv") \
        .option("header", True) \
        .option("delimiter", ",") \
        .option("escape", "\\") \
        .schema(custom_schema) \
        .load(filepath)
    return df

substitutions_bronze_df = create_source(filepath)
substitutions_bronze_df.printSchema()
display(substitutions_bronze_df)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/substitutions"
    mode_name = "overwrite"
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)

write(substitutions_bronze_df)
dbutils.fs.ls(f"{working_directory}/output/substitutions")

### **Bookings**

In [0]:
url = "https://raw.githubusercontent.com/jfjelstul/worldcup/refs/heads/master/data-csv/bookings.csv"
helpers.download_to_local_dir(url)
filepath = f"{working_directory}/bookings.csv"
dbutils.fs.ls(f"{working_directory}")

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

def create_source(filepath: str) -> DataFrame:
    custom_schema = StructType([
        StructField("key_id", IntegerType(), True),
        StructField("booking_id", StringType(), True),
        StructField("tournament_id", StringType(), True),
        StructField("tournament_name", StringType(), True),
        StructField("match_id", StringType(), True),
        StructField("match_name", StringType(), True),
        StructField("match_date", DateType(), True),
        StructField("stage_name", StringType(), True),
        StructField("group_name", StringType(), True),
        StructField("team_id", StringType(), True),
        StructField("team_name", StringType(), True),
        StructField("team_code", StringType(), True),
        StructField("home_team", IntegerType(), True),
        StructField("away_team", IntegerType(), True),
        StructField("player_id", StringType(), True),
        StructField("family_name", StringType(), True),
        StructField("given_name", StringType(), True),
        StructField("shirt_number", IntegerType(), True),
        StructField("minute_label", StringType(), True),
        StructField("minute_regulation", IntegerType(), True),
        StructField("minute_stoppage", IntegerType(), True),
        StructField("match_period", StringType(), True),
        StructField("yellow_card", IntegerType(), True),
        StructField("red_card", IntegerType(), True),
        StructField("second_yellow_card", IntegerType(), True),
        StructField("sending_off", IntegerType(), True)
    ])
    
    df = spark.read.format("csv") \
        .option("header", True) \
        .option("delimiter", ",") \
        .option("escape", "\\") \
        .schema(custom_schema) \
        .load(filepath)
    return df

bookings_bronze_df = create_source(filepath)
bookings_bronze_df.printSchema()
display(bookings_bronze_df)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/bookings"
    mode_name = "overwrite"
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)

write(bookings_bronze_df)
dbutils.fs.ls(f"{working_directory}/output/bookings")